# 🕸️ LangGraph Core Concepts

## Learning Objectives
In this notebook, you will learn:
1. **StateGraph basics** - defining a state schema and building a single-node graph
2. **Reducers** - how `Annotated[type, operator.add]` controls how state updates merge instead of overwrite
3. **Message-based state** - using `add_messages` to accumulate a conversation history
4. **Multi-node pipelines** - chaining nodes into a sequential analyze -> enhance -> finalize pipeline
5. **Building your own graph** - the exercise pattern for a two-node question/answer graph

## Prerequisites
- An OpenAI-compatible `OPENAI_API_KEY` in a `.env` file at the project root
- Familiarity with Python `TypedDict` and type hints
- `langgraph`, `langchain`, `langchain-openai`, `python-dotenv` installed

> Converted from `01_langgraph_core.py` — part of **03 LangGraph Fundamentals**.

---
## 📦 Part 0: Environment Setup

Load environment variables and import the `StateGraph`/`START`/`END` primitives, along with the message types and reducer helper (`add_messages`) used later in this notebook.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and .env credentials
# ============================================================================
import operator

from dotenv import load_dotenv
from typing_extensions import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph, add_messages

load_dotenv()

print("✅ Environment loaded and imports ready!")

---
## 🗂️ Part 1: Single-Node StateGraph

The smallest possible LangGraph: one state schema, one node, wired straight from `START` to `END`. This establishes the core mental model — a graph is a state schema plus nodes that return partial state updates — before adding reducers or multiple nodes.

### `SimpleState`

In [ ]:
# ============================================================================
# SIMPLE STATE: Input/output/step schema for the single-node demo
# ============================================================================
class SimpleState(TypedDict):
    input: str
    output: str
    step: int

### `demo_simple_graph`

Builds the graph, compiles it, and runs it once. The commented-out mermaid/PNG export lines are left as an opt-in — uncomment to visualize the graph structure locally.

In [ ]:
# ============================================================================
# DEMO: Build, Compile, and Run a Single-Node Graph
# ============================================================================
def demo_simple_graph():
    # define node functions
    def process(state: SimpleState) -> dict:
        # simple processing logic, for demo purposes
        return {"output": state["input"].upper(), "step": state["step"] + 1}

    # create graph
    graph = StateGraph(SimpleState)

    # add nodes
    graph.add_node("process", process)
    # add edges
    graph.add_edge(START, "process")
    graph.add_edge("process", END)

    # execute graph/ compile
    app = graph.compile()

    # # visualize the graph
    # print("\n--- Mermaid Graph ---")
    # print(app.get_graph().draw_mermaid())

    # # save as PNG
    # png_bytes = app.get_graph().draw_mermaid_png()
    # with open("graph.png", "wb") as f:
    #     f.write(png_bytes)
    # print("\nGraph saved to graph.png")

    # run app
    result = app.invoke({"input": "hello", "output": "", "step": 0})

    print("simple graph result:", result)
    print(
        f" Input: {result['input']}, Output: {result['output']}, Step: {result['step']}"
    )

---
## ➕ Part 2: Reducers and Accumulating State

### Key Concepts
- **Reducer**: the second type argument in `Annotated[type, reducer_fn]` tells LangGraph how to *merge* a node's returned update into existing state, instead of overwriting it.
- Here, `operator.add` concatenates lists and sums integers — so two nodes each returning `{"count": 1}` end up with `count == 2`, not `1`.

### `AccumulatingState`

In [ ]:
# ============================================================================
# ACCUMULATING STATE: A reducer-based schema using operator.add
# ============================================================================
class AccumulatingState(TypedDict):
    messages: Annotated[list[str], operator.add]  # lists concatenate when merged
    count: Annotated[int, operator.add]  # counts sum when merged

### `demo_accumulating_state`

Two nodes each contribute to `messages` and `count` — the reducer merges their outputs instead of the second node's return value clobbering the first's.

In [ ]:
# ============================================================================
# DEMO: Two Nodes Whose Updates Merge via the Reducer
# ============================================================================
def demo_accumulating_state():
    def step_one(state: AccumulatingState) -> dict:
        return {"messages": ["Step 1 executed"], "count": 1}

    def step_two(state: AccumulatingState) -> dict:
        return {"messages": ["Step 2 executed"], "count": 1}

    graph = StateGraph(AccumulatingState)

    print("\nGraph saved to graph_2.png")
    graph.add_node("step_one", step_one)
    graph.add_node("step_two", step_two)
    graph.add_edge(START, "step_one")
    graph.add_edge("step_one", "step_two")
    graph.add_edge("step_two", END)

    app = graph.compile()

    # # visualize the graph
    print("\n--- Mermaid Graph ---")
    print(app.get_graph().draw_mermaid())

    # save as PNG
    png_bytes = app.get_graph().draw_mermaid_png()
    with open("graph_2.png", "wb") as f:
        f.write(png_bytes)

    result = app.invoke({"messages": ["Initial message"], "count": 0})

    print("\nAccumulating State Result:")
    print(f"  Messages: {result['messages']}")
    print(f"  Count: {result['count']}")

---
## 💬 Part 3: Message-Based State

### Key Insight
`add_messages` is LangGraph's built-in reducer for chat history: it appends new messages to the list (and can replace an existing message by matching `id`), so a chat node only needs to return the *new* messages, not the whole running history.

### `MessageState`

In [ ]:
# ============================================================================
# MESSAGE STATE: A schema using the add_messages reducer for chat history
# ============================================================================
class MessageState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

### `demo_message_state`

A single chat node that calls the LLM with the running message list and returns its reply — `add_messages` handles appending it to history.

In [ ]:
# ============================================================================
# DEMO: A Single Chat Node Backed by add_messages
# ============================================================================
def demo_message_state():
    llm = init_chat_model("gpt-4o-mini", temperature=0)

    def chat_node(state: MessageState) -> dict:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    graph = StateGraph(MessageState)
    graph.add_node("chat_node", chat_node)
    graph.add_edge(START, "chat_node")
    graph.add_edge("chat_node", END)

    app = graph.compile()

    result = app.invoke({"messages": [HumanMessage(content="Say Hello in Tagalog")]})

    print("\nMessage State Result:")
    for msg in result["messages"]:
        role = "Human" if isinstance(msg, HumanMessage) else "AI"
        print(f"  {role}: {msg.content}")

---
## 🔗 Part 4: Multi-Node Pipelines

Chains three LLM-backed nodes sequentially — analyze -> enhance -> finalize — where each node reads the previous node's output field and writes its own. This is the graph-native equivalent of an LCEL sequential chain.

### `MultiStepState`

In [ ]:
# ============================================================================
# MULTI-STEP STATE: Schema threading input through three sequential stages
# ============================================================================
class MultiStepState(TypedDict):
    input: str
    analyzed: str
    enhanced: str
    final: str

### `demo_multi_node_graph`

In [ ]:
# ============================================================================
# DEMO: Three-Node Sequential Pipeline (analyze -> enhance -> finalize)
# ============================================================================
def demo_multi_node_graph():
    llm = init_chat_model("gpt-4o-mini", temperature=0)

    def analyze_node(state: MultiStepState) -> dict:
        response = llm.invoke(
            [
                HumanMessage(
                    content=f"Analyze the following input and summarize it in one sentence: {state['input']}"
                )
            ]
        )
        return {"analyzed": response.content}

    def enhance(state: MultiStepState) -> dict:
        response = llm.invoke(
            [
                HumanMessage(
                    content=f"Take the following analysis and enhance it with more details: {state['analyzed']}"
                )
            ]
        )

        return {"enhanced": response.content}

    def finalize(state: MultiStepState) -> dict:
        response = llm.invoke(
            [
                HumanMessage(
                    content=f"Take the following enhanced analysis and finalize it into a concise summary: {state['enhanced']}"
                )
            ]
        )
        return {"final": response.content}

    graph = StateGraph(MultiStepState)
    graph.add_node("analyze_node", analyze_node)
    graph.add_node("enhance_node", enhance)
    graph.add_node("finalize_node", finalize)

    graph.add_edge(START, "analyze_node")
    graph.add_edge("analyze_node", "enhance_node")
    graph.add_edge("enhance_node", "finalize_node")
    graph.add_edge("finalize_node", END)

    app = graph.compile()

    # # visualize the graph
    print("\n--- Mermaid Graph ---")
    print(app.get_graph().draw_mermaid())

    # save as PNG
    png_bytes = app.get_graph().draw_mermaid_png()
    with open("graph_3.png", "wb") as f:
        f.write(png_bytes)

    result = app.invoke({"input": "Artificial intelligence"})

    print("\nMulti-Node Graph Result:")
    print(f"  Input: {result['input']}")
    print(f"  Analyzed: {result['analyzed'][:100]}...")
    print(f"  Enhanced: {result['enhanced'][:100]}...")
    print(f"  Final: {result['final']}")

---
## 🏋️ Part 5: Exercise — Build Your Own Graph

### `exercise_first_langgraph`

EXERCISE: Create a LangGraph that:

In [ ]:
# ============================================================================
# EXERCISE: A Two-Node Question-Generation and Answering Graph
# ============================================================================
def exercise_first_langgraph():
    """
    EXERCISE: Create a LangGraph that:
    1. Takes a topic as input
    2. Node 1: Generates 3 questions about the topic
    3. Node 2: Answers one of the questions
    4. Returns both questions and answer
    """

    class QAState(TypedDict):
        topic: str
        questions: str
        answer: str

    llm = init_chat_model("gpt-4o-mini", temperature=0)

    def generate_questions(state: QAState) -> dict:
        response = llm.invoke(
            f"Generate 3 interesting questions about: {state['topic']}\n"
            "Format: numbered list"
        )
        return {"questions": response.content}

    def answer_question(state: QAState) -> dict:
        response = llm.invoke(
            f"Answer the first question from this list:\n{state['questions']}"
        )
        return {"answer": response.content}

    graph = StateGraph(QAState)

    graph.add_node("generate_questions", generate_questions)
    graph.add_node("answer_question", answer_question)

    graph.add_edge(START, "generate_questions")
    graph.add_edge("generate_questions", "answer_question")
    graph.add_edge("answer_question", END)

    app = graph.compile()

    result = app.invoke({"topic": "The future of renewable energy"})

    print("\nExercise Result:")
    print(f"  Topic: {result['topic']}")
    print(f"  Questions: {result['questions']}")
    print(f"  Answer: {result['answer']}")

---
## ▶️ Running the Demos

The original `__main__` guard, kept verbatim. Jupyter sets `__name__` to `"__main__"`, so this cell runs as-is. Uncomment a line to run that demo.

In [ ]:
# ============================================================================
# RUN: Execute one demo (uncomment to try others)
# ============================================================================
if __name__ == "__main__":
    # demo_simple_graph()
    # demo_accumulating_state()
    # go to .env LANGSMITH_TRACING=false to disable langsmith tracing for the next example, or set it to true to see the tracing in action
    # demo_message_state()
    # demo_multi_node_graph()
    exercise_first_langgraph()

---
## 📝 Summary

In this notebook, we learned:

### 1. State and Graph Basics
- **`StateGraph(Schema)`**: define a `TypedDict` state schema, add nodes with `add_node`, wire them with `add_edge`, and `compile()` to get a runnable graph.
- A node is just a function that takes the state and returns a *partial* update dict.

### 2. Reducers Control How State Merges
- `Annotated[type, reducer_fn]` lets two nodes contribute to the same field without one overwriting the other — `operator.add` concatenates/sums, `add_messages` appends chat messages.

### 3. Composing Nodes
- Single-node graphs are the simplest case; multi-node pipelines chain nodes sequentially, each reading the prior node's output.

### Next Steps
- Continue to `02_first_graph.ipynb` for a full conversational graph built with these same primitives.